
# Topic 2 — LoRA / QLoRA / PEFT

**Core question:** *How do I fine-tune a large model cheaply?*

**Running example:** same Amazon-style AI shopping / customer-support
assistant and the same 500-example dataset from Topic 1.

This is probably the single highest-value notebook in the series. We run
**three experiments on the same data**:

- **A — Full fine-tuning**: every weight in the model is trainable.
- **B — LoRA**: freeze the base model, train only small added low-rank
  "adapter" matrices.
- **C — QLoRA**: same adapters as LoRA, but the frozen base model is loaded
  in 4-bit quantized precision, cutting its memory footprint roughly 4x.

```mermaid
flowchart TB
    subgraph A[Full fine-tuning]
    A1[All weights trainable] --> A2[Most GPU memory + storage]
    end
    subgraph B[LoRA]
    B1[Base model frozen] --> B2[Small adapter matrices trainable]
    B2 --> B3[Adapter = a few MB]
    end
    subgraph C[QLoRA]
    C1[Base model frozen + 4-bit quantized] --> C2[Same small adapters trainable]
    C2 --> C3[Lowest GPU memory]
    end
```

### Can you answer this?
1. What's the actual difference between LoRA and QLoRA?
2. Why does the LoRA rank `r` matter, and what happens if you set it too low or too high?
3. When would you still choose full fine-tuning over LoRA?
4. What is the memory/quality trade-off QLoRA is making?


In [1]:

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()

def banner(title, subtitle="", style="bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))

def section(title):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")

def kv_table(title, rows, caption=""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 2: LoRA / QLoRA / PEFT", "Full FT vs LoRA vs QLoRA on the same customer-support dataset")


╔═══════════════════════════════════════════════════════════════╗
║ Topic 2: LoRA / QLoRA / PEFT                                  ║
║ Full FT vs LoRA vs QLoRA on the same customer-support dataset ║
╚═══════════════════════════════════════════════════════════════╝


## Concept primer

**PEFT (Parameter-Efficient Fine-Tuning)** is a family of methods that train
only a small added set of parameters instead of every weight in the model.
`peft` is Hugging Face's library implementing these methods.

**Low-rank matrices / LoRA**: instead of updating a full weight matrix `W`
(shape `d x k`, potentially millions of numbers), LoRA freezes `W` and adds a
*low-rank* update `B @ A` where `B` is `d x r` and `A` is `r x k`, with `r`
("rank") much smaller than `d` or `k` (e.g. r=16). Only `A` and `B` are
trained — a tiny fraction of the original parameter count — because the
*hypothesis* behind LoRA is that the weight *change* needed for a new task
lives in a low-dimensional subspace, even if the original weight matrix is huge.

**Trainable vs. frozen parameters**: in full fine-tuning, 100% of parameters
are trainable. In LoRA/QLoRA, the base model's parameters are frozen
(`requires_grad=False`) and only the adapter matrices are trainable.

**Rank `r`**: the "width" of the adapter. Too low (e.g. r=1-2) and the
adapter may not have enough capacity to capture the needed behavior change.
Too high (approaching the full dimension) and you lose the efficiency
benefit while rarely gaining much quality past a point of diminishing returns.

**LoRA alpha**: a scaling factor applied to the adapter's output
(`scaling = alpha / r`). It controls how strongly the adapter's update
influences the frozen base output — conceptually similar to a learning-rate
multiplier specifically for the adapter path.

**Target modules**: which weight matrices get a LoRA adapter attached —
typically the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`)
and often the MLP projections too. More target modules = more trainable
parameters = more capacity, but also more memory/compute.

**Quantization / 4-bit training (QLoRA)**: the frozen base model's weights
are stored in 4-bit precision (via `bitsandbytes`) instead of 16/32-bit —
roughly a 4x memory reduction for the (much larger) frozen portion of the
model, while the small trainable LoRA adapters still train in higher
precision for numerical stability. This is what makes it possible to
fine-tune much larger models on a single consumer/Colab GPU.

**Adapter merging**: once trained, a LoRA adapter's weights can be
mathematically folded back into the base model's weights (`W' = W + B@A`),
producing a single ordinary model with no runtime overhead — useful for
final deployment. Until merged, you can also run **multiple adapters**
side-by-side on one frozen base model (e.g. one per customer or per task)
and swap between them cheaply, since each adapter is only a few MB.


In [2]:

import torch
section("Environment check")
HAS_CUDA = torch.cuda.is_available()
kv_table("Runtime", {"CUDA available": HAS_CUDA, "PyTorch": torch.__version__})
if not HAS_CUDA:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected.[/bold yellow] Structural comparisons below "
        "(parameter counts, adapter size, estimated memory) are computed for real without "
        "needing a GPU. Actual training time and validation score require a GPU -- run on "
        "Colab (T4) to get those.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

          Runtime          
                           
  Key              Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━ 
  CUDA available   False   
  PyTorch          2.14.1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected. Structural comparisons below (parameter counts, adapter size, estimated memory) are       ┃
┃ computed for real without needing a GPU. Actual training time and validation score require a GPU -- run on      ┃
┃ Colab (T4) to get those.                                                                                        ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

import sys
IN_COLAB = "google.colab" in sys.modules
packages = ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "torch", "datasets", "rich"]
if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages)}")
    console.print("[green]Installed packages for Colab.[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv already has these.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these.

In [4]:
# --- Load the dataset (reused from Topic 1) ---------------------------------
import json
import random
from pathlib import Path
from collections import Counter

section("Loading the shared dataset")

vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]

RELATIVE_CANDIDATES = ["data/support_tickets.json", "02_lora_qlora_peft/data/support_tickets.json",
                        "01_sft_fundamentals/data/support_tickets.json"]
DATA_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH:
        break

if DATA_PATH is None:
    # Self-sufficient fallback: regenerate the exact same deterministic dataset
    # in-memory instead of requiring a separate file upload (e.g. on Colab).
    console.print("[dim]support_tickets.json not found nearby -- generating it in-memory "
                  "instead of requiring a separate file upload.[/dim]")

    PRODUCTS = ["wireless earbuds", "coffee maker", "running shoes", "laptop stand",
                "desk lamp", "yoga mat", "bluetooth speaker", "phone case",
                "winter jacket", "kitchen blender", "backpack", "office chair",
                "water bottle", "gaming mouse", "electric toothbrush", "air fryer",
                "board game", "children's book", "garden hose", "car phone mount"]
    NAMES = ["Alex", "Jordan", "Priya", "Sam", "Taylor", "Morgan", "Chris",
             "Jamie", "Riley", "Casey", "Drew", "Avery"]

    def _order(): return f"#{random.randint(100000, 999999)}"
    def _days(lo=1, hi=14): return random.randint(lo, hi)

    def order_status():
        p, o, d = random.choice(PRODUCTS), _order(), _days(1, 10)
        return (f"I ordered a {p} with order number {o} {d} days ago and I still don't see any "
                f"shipping update. Can you tell me what's going on?",
                f"Thanks for reaching out about order {o}. I checked and your {p} is currently being "
                f"prepared at our fulfillment center and is on track to ship within 1-2 business days. "
                f"You'll receive a tracking email the moment it ships, and estimated delivery is "
                f"{_days(2,5)} days after that. I'll flag the order so it's monitored closely.")

    def returns_refunds():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I want to return the {p} from order {o} because it's not what I expected. How do I "
                f"get a refund?",
                f"No problem at all -- I've started a return for the {p} in order {o}. You'll get a "
                f"prepaid return label by email within the hour. Once the item is scanned by the "
                f"carrier, your refund of the full purchase price will be issued to your original "
                f"payment method within 3-5 business days.")

    def damaged_item():
        p, o = random.choice(PRODUCTS), _order()
        return (f"The {p} I received in order {o} arrived damaged -- it looks like it was crushed in "
                f"shipping. What can you do?",
                f"I'm sorry the {p} arrived damaged, that's not the experience we want for you. I've "
                f"issued a free replacement for order {o} at no extra cost, and you don't need to "
                f"return the damaged item. The replacement will ship within 1 business day with "
                f"expedited shipping.")

    def payment_issue():
        o = _order()
        amount = random.choice([19.99, 29.99, 49.50, 89.00, 124.75])
        return (f"I was charged ${amount} twice for order {o}. Can you check my payment and fix this?",
                f"I checked order {o} and confirmed a duplicate charge of ${amount}. I've submitted a "
                f"refund for the extra charge, which will appear on your statement within 3-5 business "
                f"days. Only one charge of ${amount} will remain for your order.")

    def account_access():
        name = random.choice(NAMES)
        return (f"Hi, this is {name}. I can't log into my account, it keeps saying my password is "
                f"incorrect even after I reset it. Can you help?",
                f"Hi {name}, sorry for the trouble logging in. I've sent a fresh password reset link "
                f"to your account email -- it will be valid for 30 minutes. If it still doesn't work "
                f"after that, it may be a browser cache issue, so try an incognito window or clearing "
                f"cookies for our site.")

    def shipping_delay():
        p, o, d = random.choice(PRODUCTS), _order(), _days(5, 14)
        return (f"My {p} from order {o} was supposed to arrive {d} days ago and it still hasn't shown "
                f"up. This is really frustrating.",
                f"I completely understand the frustration, and I'm sorry for the delay on your {p} in "
                f"order {o}. I checked with the carrier and the package is delayed in transit but "
                f"still moving -- updated delivery is within 2 days. As an apology for the delay, I've "
                f"credited your account with a $10 voucher for your next order.")

    def cancellation():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I need to cancel my order {o} for the {p} -- I accidentally ordered two of them. Is "
                f"it too late?",
                f"Good news -- order {o} hasn't shipped yet, so I was able to cancel it for you right "
                f"away. You will not be charged for this order, and any pending authorization will "
                f"drop off within a few business days. The other order for the {p} is unaffected.")

    def product_question():
        p = random.choice(PRODUCTS)
        return (f"Before I buy the {p}, can you tell me if it comes with a warranty and what the "
                f"return window is?",
                f"Great question! The {p} comes with a 1-year manufacturer warranty covering defects, "
                f"and you can return it within 30 days of delivery for any reason as long as it's in "
                f"like-new condition. If anything goes wrong after the return window, our support "
                f"team can still help with the manufacturer warranty.")

    def subscription_issue():
        return ("I signed up for a monthly subscription but I was charged again this month even "
                "though I thought I canceled it last week. Can you check?",
                "I looked into your subscription and I can confirm the cancellation request from "
                "last week did not fully process due to a billing sync issue on our end. I've now "
                "canceled the subscription effective immediately and refunded this month's charge "
                "in full -- it will reflect within 3-5 business days.")

    def promo_discount():
        code = random.choice(["SAVE10", "WELCOME15", "SUMMER20", "FIRST5"])
        return (f"I tried to use the promo code {code} at checkout but it says invalid. The order "
                f"total didn't get discounted. Can you help?",
                f"Thanks for flagging this. I checked the promo code {code} and applied the discount "
                f"manually to your most recent order, so you'll see the adjusted total reflected "
                f"shortly. In the future, that code works best when applied before adding a gift wrap "
                f"option, which was conflicting with it at checkout.")

    CATEGORIES = {
        "order_status": order_status, "returns_refunds": returns_refunds,
        "damaged_item": damaged_item, "payment_issue": payment_issue,
        "account_access": account_access, "shipping_delay": shipping_delay,
        "cancellation": cancellation, "product_question": product_question,
        "subscription_issue": subscription_issue, "promo_discount": promo_discount,
    }

    def generate_support_tickets(n=500, seed=42):
        rng_state = random.getstate()
        random.seed(seed)
        names = list(CATEGORIES.keys())
        per, rem = n // len(names), n % len(names)
        examples = []
        for i, name in enumerate(names):
            count = per + (1 if i < rem else 0)
            for _ in range(count):
                problem, response = CATEGORIES[name]()
                examples.append({"category": name, "problem": problem, "response": response})
        random.shuffle(examples)
        random.setstate(rng_state)
        return examples

    raw_examples = generate_support_tickets()
    DATA_PATH = Path.cwd() / "data" / "support_tickets.json"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_PATH.open("w") as f:
        json.dump(raw_examples, f, indent=2)
else:
    with DATA_PATH.open() as f:
        raw_examples = json.load(f)

console.print(f"Loaded [bold]{len(raw_examples)}[/bold] examples from {DATA_PATH} (same dataset as Topic 1)")


── Loading the shared dataset ──────────────────────────────────

Loaded 500 examples from 
/Users/bhanupraka.gopularam/aiprojects/llm-fine-tuning/02_lora_qlora_peft/data/support_tickets.json (same dataset 
as Topic 1)


## Building the three model variants

We use `Qwen/Qwen3-0.6B`'s **architecture config** to build each variant.
Note: building from config (`from_config`) constructs the model with the
correct architecture and parameter *shapes* instantly, with no weight
download needed -- perfect for counting parameters and estimating memory.
A real training run (GPU-only, below) instead loads the actual pretrained
weights via Unsloth.


In [5]:

# --- Build Full FT / LoRA / QLoRA variants and count parameters -------------
from transformers import AutoConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-0.6B"
section("Building model variants (architecture only -- no weight download needed)")

config = AutoConfig.from_pretrained(MODEL_NAME)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# A: Full fine-tuning -- every parameter is trainable
full_ft_model = AutoModelForCausalLM.from_config(config)
full_total, full_trainable = count_params(full_ft_model)

# B: LoRA -- freeze base, attach small trainable adapters
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
LORA_R = 16
LORA_ALPHA = 32
lora_base_model = AutoModelForCausalLM.from_config(config)
lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
                       lora_dropout=0.0, task_type="CAUSAL_LM")
lora_model = get_peft_model(lora_base_model, lora_cfg)
lora_total, lora_trainable = count_params(lora_model)

# C: QLoRA -- identical adapter config to LoRA; the difference is the frozen
# base model is stored in 4-bit instead of 16/32-bit. Loading in real 4-bit
# requires bitsandbytes CUDA kernels, so trainable/adapter counts are
# identical to LoRA by construction -- only the *base model memory* differs,
# which we estimate below rather than measure.
qlora_trainable = lora_trainable
qlora_total = lora_total

kv_table("Parameter counts (real, computed from the actual Qwen3-0.6B architecture)", {
    "Full FT -- trainable": f"{full_trainable:,} / {full_total:,} (100%)",
    "LoRA -- trainable": f"{lora_trainable:,} / {lora_total:,} ({lora_trainable/lora_total*100:.2f}%)",
    "QLoRA -- trainable": f"{qlora_trainable:,} / {qlora_total:,} ({qlora_trainable/qlora_total*100:.2f}%) (same adapters as LoRA)",
})


W1002 20:42:43.913000 32584 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


W1002 20:42:43.946000 32584 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1002 20:42:43.977000 32584 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


── Building model variants (architecture only -- no weight download needed) 

    Parameter counts (real, computed from the actual Qwen3-0.6B architecture)     
                                                                                  
  Key                    Value                                                    
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Full FT -- trainable   596,049,920 / 596,049,920 (100%)                         
  LoRA -- trainable      4,587,520 / 600,637,440 (0.76%)                          
  QLoRA -- trainable     4,587,520 / 600,637,440 (0.76%) (same adapters as LoRA)

In [6]:

# --- Estimate GPU memory + adapter size from the real parameter counts ------
section("Estimated GPU memory & adapter size (computed from real parameter counts)")

BYTES_FP32, BYTES_FP16, BYTES_INT4 = 4, 2, 0.5

def adamw_training_memory_bytes(trainable_params, frozen_params, frozen_bytes_per_param, trainable_bytes_per_param=2):
    # Trainable params need: weights + gradients + 2 AdamW optimizer moments (all ~fp16/bf16 here)
    trainable_mem = trainable_params * trainable_bytes_per_param * 4
    # Frozen params only need to be stored (no grad, no optimizer state)
    frozen_mem = frozen_params * frozen_bytes_per_param
    return trainable_mem + frozen_mem

full_mem = adamw_training_memory_bytes(full_trainable, 0, BYTES_FP16, trainable_bytes_per_param=BYTES_FP16)
# Full FT here trains everything in fp16-ish mixed precision for a fair apples-to-apples estimate
lora_frozen = lora_total - lora_trainable
lora_mem = adamw_training_memory_bytes(lora_trainable, lora_frozen, BYTES_FP16)
qlora_mem = adamw_training_memory_bytes(qlora_trainable, lora_frozen, BYTES_INT4)

adapter_size_mb = lora_trainable * BYTES_FP16 / 1e6

table = Table(title="Full FT vs LoRA vs QLoRA -- Topic 2 experiment results", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Metric")
table.add_column("Full FT (A)", style="red")
table.add_column("LoRA (B)", style="yellow")
table.add_column("QLoRA (C)", style="green")
table.add_row("GPU memory (estimated, training)", f"~{full_mem/1e9:.2f} GB", f"~{lora_mem/1e9:.2f} GB", f"~{qlora_mem/1e9:.2f} GB")
table.add_row("Trainable parameters (real)", f"{full_trainable:,}", f"{lora_trainable:,}", f"{qlora_trainable:,}")
table.add_row("Training time", "[dim]measured on GPU -- see below[/dim]", "[dim]measured on GPU -- see below[/dim]", "[dim]measured on GPU -- see below[/dim]")
table.add_row("Validation score", "[dim]measured on GPU -- see below[/dim]", "[dim]measured on GPU -- see below[/dim]", "[dim]measured on GPU -- see below[/dim]")
table.add_row("Adapter size on disk", "N/A (full model)", f"~{adapter_size_mb:.1f} MB", f"~{adapter_size_mb:.1f} MB (same adapter)")
console.print(table)

console.print(
    "\n[dim]Memory estimates use a simple AdamW accounting model: trainable params cost "
    "weight+grad+2 optimizer moments (~4x their dtype size), frozen params only cost storage "
    "at their dtype size (fp16 for LoRA's base, int4 for QLoRA's base). Real measured memory "
    "on a GPU will differ due to activations, KV cache, and framework overhead -- this is a "
    "back-of-envelope estimate to make the *relative* difference concrete.[/dim]"
)


── Estimated GPU memory & adapter size (computed from real parameter counts) 

                              Full FT vs LoRA vs QLoRA -- Topic 2 experiment results                               
                                                                                                                   
  Metric                       Full FT (A)                 LoRA (B)                     QLoRA (C)                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  GPU memory (estimated,       ~4.77 GB                    ~1.23 GB                     ~0.33 GB                   
  training)                                                                                                        
                                                                                                                   
  Trainable parameters         596,049,920                 4,587,520                    4,587,520                  
  (real)                                                                                                           
                                                                                                                   
  Training time                measured on GPU -- see      measured on GPU -- see       measured on GPU -- see     
                               below                       below                        below                      
                                                                                                                   
  Validation score             measured on GPU -- see      measured on GPU -- see       measured on GPU -- see     
                               below                       below                        below                      
                                                                                                                   
  Adapter size on disk         N/A (full model)            ~9.2 MB                      ~9.2 MB (same adapter)    

Memory estimates use a simple AdamW accounting model: trainable params cost weight+grad+2 optimizer moments (~4x 
their dtype size), frozen params only cost storage at their dtype size (fp16 for LoRA's base, int4 for QLoRA's 
base). Real measured memory on a GPU will differ due to activations, KV cache, and framework overhead -- this is a 
back-of-envelope estimate to make the *relative* difference concrete.


## Running the real experiments on a GPU

Each training run below reuses the same 490 training / 10 held-out split as
Topic 1, 3 epochs, batch size 4. We measure **actual** peak GPU memory,
wall-clock training time, and a simple keyword-overlap validation score
(fraction of expected key phrases from the target response present in the
model's generated response on held-out problems) -- not a rigorous judge,
but enough to see the *relative* ordering between the three methods.


In [7]:

# --- Real experiments (GPU-only) --------------------------------------------
import random, time

random.seed(7)
shuffled = raw_examples[:]
random.shuffle(shuffled)
N_HOLDOUT = 10
holdout_examples = shuffled[:N_HOLDOUT]
train_examples = shuffled[N_HOLDOUT:]

SYSTEM_PROMPT = ("You are a helpful customer support assistant for an online shopping "
                  "platform. Respond to the customer's problem clearly and helpfully.")

def to_conversation(example):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["response"]},
    ]

def keyword_overlap_score(generated, expected, k=6):
    expected_words = [w.strip(".,!?$#").lower() for w in expected.split() if len(w) > 4]
    key_terms = list(dict.fromkeys(expected_words))[:k]
    if not key_terms:
        return 0.0
    hits = sum(1 for term in key_terms if term in generated.lower())
    return hits / len(key_terms)

def run_experiment(name, use_lora, use_4bit):
    from unsloth import FastLanguageModel
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=512, load_in_4bit=use_4bit,
    )
    if use_lora:
        model = FastLanguageModel.get_peft_model(
            model, r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
        )

    def to_text(ex):
        return {"text": tok.apply_chat_template(to_conversation(ex), tokenize=False)}
    train_ds = Dataset.from_list([to_text(ex) for ex in train_examples])

    FastLanguageModel.for_training(model)
    cfg = SFTConfig(output_dir=f"outputs/{name}", num_train_epochs=3,
                     per_device_train_batch_size=4, gradient_accumulation_steps=2,
                     learning_rate=2e-4, max_length=512, logging_steps=50,
                     save_strategy="no", report_to=[])
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, args=cfg)
    result = trainer.train()

    FastLanguageModel.for_inference(model)
    scores = []
    for ex in holdout_examples[:5]:
        conv = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": ex["problem"]}]
        prompt_text = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt_text, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        gen = tok.decode(out[0], skip_special_tokens=True)
        scores.append(keyword_overlap_score(gen, ex["response"]))

    elapsed = time.time() - t0
    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
    return {
        "training_time_s": round(elapsed, 1),
        "peak_gpu_mem_gb": round(peak_mem_gb, 2),
        "val_score": round(sum(scores) / len(scores), 2) if scores else None,
        "final_loss": round(result.training_loss, 3),
    }

if HAS_CUDA:
    section("Running real experiments -- this trains 3 small models back-to-back")
    results = {}
    results["Full FT"] = run_experiment("full_ft", use_lora=False, use_4bit=False)
    results["LoRA"] = run_experiment("lora", use_lora=True, use_4bit=False)
    results["QLoRA"] = run_experiment("qlora", use_lora=True, use_4bit=True)

    table = Table(title="Measured results (real GPU run)", box=box.SIMPLE_HEAVY, show_lines=True)
    table.add_column("Metric")
    for name in results:
        table.add_column(name)
    for metric in ["training_time_s", "peak_gpu_mem_gb", "val_score", "final_loss"]:
        table.add_row(metric, *[str(results[name][metric]) for name in results])
    console.print(table)
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- running the three real training experiments requires a CUDA GPU.\n"
        "Expected on Colab (T4): three short training runs (~1-2 min each for this small\n"
        "dataset/model), after which Full FT should show the highest GPU memory and QLoRA the\n"
        "lowest, while validation keyword-overlap scores should land in a similar range for\n"
        "all three (LoRA/QLoRA shouldn't meaningfully lag Full FT in quality on a task this small).",
        title="Real experiments (GPU-only)", style="yellow",
    ))


╭────────────────────────────────────────── Real experiments (GPU-only) ──────────────────────────────────────────╮
│ SKIPPED -- running the three real training experiments requires a CUDA GPU.                                     │
│ Expected on Colab (T4): three short training runs (~1-2 min each for this small                                 │
│ dataset/model), after which Full FT should show the highest GPU memory and QLoRA the                            │
│ lowest, while validation keyword-overlap scores should land in a similar range for                              │
│ all three (LoRA/QLoRA shouldn't meaningfully lag Full FT in quality on a task this small).                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Can you answer this? (Topic 2 training checkpoints)

**1. What's the actual difference between LoRA and QLoRA?**
<details><summary>Answer</summary>
The trainable adapters are identical in both. The difference is entirely in
how the *frozen* base model is stored during training: LoRA keeps it at full
precision (fp16/bf16), QLoRA quantizes it to 4-bit. QLoRA trades a little
numerical precision in the frozen weights for a large (~4x) reduction in the
memory needed to even hold the base model during training.
</details>

**2. Why does the LoRA rank `r` matter, and what happens if you set it too low or too high?**
<details><summary>Answer</summary>
`r` controls how much capacity the adapter has to represent the needed
weight change. Too low and the adapter can't capture the behavior shift
(underfits); too high and you approach full fine-tuning's parameter count
and lose the efficiency benefit, usually without a proportional quality
gain once `r` is already large enough for the task.
</details>

**3. When would you still choose full fine-tuning over LoRA?**
<details><summary>Answer</summary>
When the task requires changing the model's behavior so broadly/deeply that
a low-rank update can't represent it well (e.g. teaching genuinely new
capabilities or heavily rewriting internal representations), or when you
have the GPU budget and want to squeeze out the last bit of quality and
don't need to deploy many task-specific variants cheaply.
</details>

**4. What is the memory/quality trade-off QLoRA is making?**
<details><summary>Answer</summary>
QLoRA sacrifices a small amount of numerical precision in the frozen base
model's weights (4-bit vs 16-bit) in exchange for a large memory reduction,
which is what enables fine-tuning much larger models on limited hardware.
In practice, quality loss from the 4-bit base is usually small because only
the *adapters* are being optimized -- the base model's quantization noise is
mostly "along for the ride."
</details>


In [8]:

section("Scale discussion: LoRA/QLoRA at ~100 vs ~10k users")
table = Table(title="LoRA/QLoRA adapters at scale", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Concern")
table.add_column("~100 users", style="cyan")
table.add_column("~10k users", style="magenta")
rows = [
    ("Adapter storage", "A few MB per adapter -- trivial to store many versions",
                        "Still trivial -- storage was never the bottleneck; the win compounds as you accumulate many task/customer-specific adapters"),
    ("Serving multiple variants", "Can load one adapter at a time on a single GPU",
                                  "vLLM and similar servers support multi-LoRA serving -- many adapters hot-swapped on one base model deployment, instead of one full model copy per variant"),
    ("Retraining cost", "A LoRA/QLoRA retrain is cheap enough to do ad hoc",
                        "Still cheap per adapter -- the constraint becomes eval/review pipeline throughput, not compute"),
    ("Base model updates", "Re-running LoRA against a new base model version is fast",
                           "Need a rollout plan: re-train/validate all adapters against the new base before switching production traffic"),
]
for c, s, b in rows:
    table.add_row(c, s, b)
console.print(table)


── Scale discussion: LoRA/QLoRA at ~100 vs ~10k users ──────────

                                           LoRA/QLoRA adapters at scale                                            
                                                                                                                   
  Concern                     ~100 users                                 ~10k users                                
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Adapter storage             A few MB per adapter -- trivial to store   Still trivial -- storage was never the    
                              many versions                              bottleneck; the win compounds as you      
                                                                         accumulate many task/customer-specific    
                                                                         adapters                                  
                                                                                                                   
  Serving multiple variants   Can load one adapter at a time on a        vLLM and similar servers support          
                              single GPU                                 multi-LoRA serving -- many adapters       
                                                                         hot-swapped on one base model             
                                                                         deployment, instead of one full model     
                                                                         copy per variant                          
                                                                                                                   
  Retraining cost             A LoRA/QLoRA retrain is cheap enough to    Still cheap per adapter -- the            
                              do ad hoc                                  constraint becomes eval/review pipeline   
                                                                         throughput, not compute                   
                                                                                                                   
  Base model updates          Re-running LoRA against a new base model   Need a rollout plan: re-train/validate    
                              version is fast                            all adapters against the new base before  
                                                                         switching production traffic             

In [9]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Reused the Topic 1 dataset\n"
    "  2. Built Full FT / LoRA / QLoRA model variants and counted REAL trainable parameters\n"
    "  3. Computed estimated GPU memory and adapter size from those real counts\n"
    "  4. (On GPU) ran all three as real training experiments and measured memory/time/quality\n"
    "  5. Discussed how adapters change the economics of serving many fine-tuned variants at scale\n\n"
    "[bold]Next: Topic 3 -- Continued Pretraining / Domain Adaptation[/bold]\n"
    "We switch from Q&A-style SFT to raw-text next-token prediction on a technical\n"
    "corpus, and compare a base model against a domain-adapted one.",
    title="Topic 2 complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔═══════════════════════════════════════════════ Topic 2 complete ════════════════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Reused the Topic 1 dataset                                                                                 ║
║   2. Built Full FT / LoRA / QLoRA model variants and counted REAL trainable parameters                          ║
║   3. Computed estimated GPU memory and adapter size from those real counts                                      ║
║   4. (On GPU) ran all three as real training experiments and measured memory/time/quality                       ║
║   5. Discussed how adapters change the economics of serving many fine-tuned variants at scale                   ║
║                                                                                                                 ║
║ Next: Topic 3 -- Continued Pretraining / Domain Adaptation                                                      ║
║ We switch from Q&A-style SFT to raw-text next-token prediction on a technical                                   ║
║ corpus, and compare a base model against a domain-adapted one.                                                  ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝